# Análise de sensibilidade do pipeline

Avalia, no conjunto de validação, a sensibilidade do pipeline à alteração de
um parâmetro por vez (OFAT). As saídas principais são o Dice da segmentação e
o sucesso dos óstios, considerando casos corretos ou toleráveis. O conjunto
de teste não participa da seleção dos parâmetros.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT_CANDIDATES = (CURRENT_DIR, *CURRENT_DIR.parents)
REPO_ROOT = next(
    root for root in REPO_ROOT_CANDIDATES if (root / "src" / "utils").is_dir()
)
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from utils.experiments import (  # noqa: E402
    build_parameter_pairwise_summary,
    build_parameter_sensitivity_summary,
    load_parameter_validation_run,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

## Configuração

O notebook combina o run principal P99.9 com o run histórico que contém P99.7
e P99.5. Todos os resultados devem pertencer ao mesmo conjunto de validação.

In [ ]:
# Define os runs quantitativos analisados.
ANALYSIS_DIR = REPO_ROOT / "output/segmentation/analysis/pipeline_parameter_validation"
RUNS_DIR = ANALYSIS_DIR / "runs"
RUN_NAME = "sensitivity_cbeb_p999_val_270"
THRESHOLD_RUN_NAME = "sensitivity_selected_val_270"

In [ ]:
# Carrega o run P99.9 e o run histórico que contém P99.7/P99.5.
RUN_DIR, primary_results, primary_parameters, run_config = load_parameter_validation_run(
    RUNS_DIR / RUN_NAME,
    expected_split="val",
)
(
    THRESHOLD_RUN_DIR,
    threshold_results,
    threshold_parameters,
    threshold_run_config,
) = load_parameter_validation_run(
    RUNS_DIR / THRESHOLD_RUN_NAME,
    expected_split="val",
)

# O baseline histórico representa P99.7; P99.9 já é o baseline do run principal.
threshold_aliases = {"baseline": "upper_p997", "upper_p995": "upper_p995"}
threshold_results = threshold_results.loc[
    threshold_results["variant"].isin(threshold_aliases)
].copy()
threshold_results["variant"] = threshold_results["variant"].replace(threshold_aliases)

threshold_parameters = threshold_parameters.loc[
    threshold_parameters["variant"].isin(threshold_aliases)
].copy()
threshold_parameters["variant"] = threshold_parameters["variant"].replace(threshold_aliases)
threshold_parameters.loc[
    threshold_parameters["variant"] == "upper_p997", "parameter_group"
] = "upper_percentile"
threshold_parameters.loc[
    threshold_parameters["variant"] == "upper_p997", "description"
] = "Percentil superior do threshold = 99.7."

results_df = pd.concat([primary_results, threshold_results], ignore_index=True)
parameters_df = pd.concat([primary_parameters, threshold_parameters], ignore_index=True)

results_df.drop(columns=["cohort_kind", "cohort_roles"], errors="ignore", inplace=True)

primary_ids = set(primary_results["IMG_ID"])
threshold_ids = set(threshold_results["IMG_ID"])
if primary_ids != threshold_ids:
    raise ValueError("Os runs P99.9, P99.7 e P99.5 não usam os mesmos exames.")

print(f"Run principal: {RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Run dos thresholds: {THRESHOLD_RUN_DIR.relative_to(REPO_ROOT)}")
print(f"Imagens: {results_df['IMG_ID'].nunique()} | Variantes: {results_df['variant'].nunique()}")

results_df.head()

## Parâmetros efetivamente avaliados

Os níveis abaixo são extraídos dos CSVs dos runs carregados. Dessa forma, a
tabela não declara como avaliado um parâmetro que apenas estava planejado, mas
cujo experimento não foi executado.

In [ ]:
# Relaciona cada grupo OFAT à coluna que contém seu valor numérico.
PARAMETER_SPECS = {
    "upper_percentile": ("Percentil superior", "MAX_THRESHOLD_PERCENTILE"),
    "ostia_z_limit": (
        "Limite z do segundo óstio (mm)",
        "OSTIA_DETECTION.max_z_diff_mm",
    ),
    "rg_threshold_divisor": (
        "Divisor global do RG",
        "REGION_GROWING.threshold_divisor",
    ),
    "rg_vesselness_floor": (
        "Fração mínima de vesselness",
        "REGION_GROWING.min_vesselness_fraction",
    ),
}

baseline_parameters = parameters_df.loc[
    parameters_df["variant"].eq("baseline")
].iloc[0]
evaluated_parameter_rows = []

for parameter_group, (parameter_label, value_column) in PARAMETER_SPECS.items():
    group_parameters = parameters_df.loc[
        parameters_df["parameter_group"].eq(parameter_group)
    ]
    if group_parameters.empty:
        continue

    # Inclui o nível de referência junto aos níveis alterados do mesmo parâmetro.
    evaluated_values = pd.concat(
        [
            pd.Series([baseline_parameters[value_column]]),
            group_parameters[value_column],
        ],
        ignore_index=True,
    ).dropna()
    evaluated_values = sorted(pd.unique(evaluated_values.astype(float)))
    evaluated_parameter_rows.append(
        {
            "parameter_group": parameter_group,
            "parameter": parameter_label,
            "reference_value": float(baseline_parameters[value_column]),
            "evaluated_levels": ", ".join(f"{value:g}" for value in evaluated_values),
            "evaluated_variants": int(group_parameters["variant"].nunique()),
        }
    )

evaluated_parameters_df = pd.DataFrame(evaluated_parameter_rows)
evaluated_parameters_df

## Resultados quantitativos

A tabela abaixo contém diretamente as medidas relevantes para a análise de
sensibilidade: número e taxa de sucessos dos óstios, além de média, desvio
padrão e mediana do Dice. O sucesso aceita localizações
classificadas como corretas ou toleráveis.

In [ ]:
# Resume Dice e sucesso dos óstios para cada configuração.
sensitivity_summary = build_parameter_sensitivity_summary(
    results_df, parameters_df, baseline_variant="baseline"
)
article_sensitivity_table = sensitivity_summary[[
    "variant", "description", "images",
    "ostia_success_count", "ostia_success_percent",
    "mean_dice", "std_dice", "median_dice",
    "delta_dice_vs_baseline",
]]
article_sensitivity_table.round(4)

### Efeito em relação à referência

Além do delta absoluto de Dice, a tabela apresenta sua variação relativa e a
mudança na taxa de sucesso dos óstios em pontos percentuais. Isso permite
distinguir alterações pequenas de segmentação de mudanças na localização.

In [ ]:
baseline_summary = sensitivity_summary.loc[
    sensitivity_summary["variant"].eq("baseline")
].iloc[0]

relative_effect_df = sensitivity_summary.copy()
relative_effect_df["relative_dice_change_percent"] = 100 * (
    relative_effect_df["mean_dice"] - baseline_summary["mean_dice"]
) / baseline_summary["mean_dice"]
relative_effect_df["delta_ostia_success_pp"] = (
    relative_effect_df["ostia_success_percent"]
    - baseline_summary["ostia_success_percent"]
)

relative_effect_columns = [
    "variant",
    "description",
    "mean_dice",
    "delta_dice_vs_baseline",
    "relative_dice_change_percent",
    "ostia_success_percent",
    "delta_ostia_success_pp",
]
relative_effect_df[relative_effect_columns].round(4)

### Amplitude de sensibilidade por parâmetro

A amplitude é a diferença entre o maior e o menor resultado observado entre o
baseline e os níveis testados de cada parâmetro. Valores pequenos indicam maior
robustez dentro da faixa avaliada; essa interpretação não deve ser extrapolada
para valores fora da grade.

In [ ]:
parameter_range_rows = []

for parameter in evaluated_parameters_df.itertuples(index=False):
    group_variants = parameters_df.loc[
        parameters_df["parameter_group"].eq(parameter.parameter_group), "variant"
    ].unique()
    compared_variants = ["baseline", *group_variants]
    group_summary = relative_effect_df.loc[
        relative_effect_df["variant"].isin(compared_variants)
    ].copy()

    largest_effect_row = group_summary.loc[
        group_summary["delta_dice_vs_baseline"].abs().idxmax()
    ]
    parameter_range_rows.append(
        {
            "parameter": parameter.parameter,
            "evaluated_levels": parameter.evaluated_levels,
            "dice_range": group_summary["mean_dice"].max()
            - group_summary["mean_dice"].min(),
            "ostia_success_range_pp": group_summary["ostia_success_percent"].max()
            - group_summary["ostia_success_percent"].min(),
            "largest_effect_variant": largest_effect_row["variant"],
            "largest_absolute_delta_dice": abs(
                largest_effect_row["delta_dice_vs_baseline"]
            ),
        }
    )

parameter_range_df = pd.DataFrame(parameter_range_rows).sort_values(
    "largest_absolute_delta_dice", ascending=False
)
parameter_range_df.round(4)

### Dice médio e variabilidade por configuração

In [ ]:
# Ordena as variantes antes de desenhar o Dice médio.
plot_df = sensitivity_summary.sort_values("mean_dice", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["mean_dice"],
    xerr=plot_df["std_dice"].fillna(0),
    color="#2878B5",
    alpha=0.9,
    capsize=3,
)
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
ax.set_xlabel("Dice Score médio", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 1)
fig.tight_layout()
plt.show()

### Taxa de sucesso na localização dos óstios

In [ ]:
# Mostra a sensibilidade da localização dos óstios aos parâmetros.
plot_df = sensitivity_summary.sort_values("ostia_success_percent", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    plot_df["variant"],
    plot_df["ostia_success_percent"],
    color="#3A923A",
    alpha=0.9,
)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=10)
ax.set_xlabel("Óstios corretos ou toleráveis (%)", fontsize=13)
ax.set_ylabel("Configuração", fontsize=13)
ax.set_xlim(0, 100)
fig.tight_layout()
plt.show()

## Alterações pareadas em relação à referência

Como todas as configurações usam os mesmos IDs, a diferença de Dice é calculada
exame a exame. Essa tabela ajuda a distinguir uma mudança sistemática de uma
média dominada por poucos casos.

In [ ]:
# Compara cada variante com o baseline nos mesmos exames.
pairwise_df = build_parameter_pairwise_summary(
    results_df, baseline_variant="baseline"
)
pairwise_df["improved_percent"] = 100 * (
    pairwise_df["improved_images"] / pairwise_df["paired_images"]
)
pairwise_df["unchanged_percent"] = 100 * (
    pairwise_df["unchanged_images"] / pairwise_df["paired_images"]
)
pairwise_df["worse_percent"] = 100 * (
    pairwise_df["worse_images"] / pairwise_df["paired_images"]
)
pairwise_df.round(4)

## Relação entre Dice e sucesso dos óstios

Cada ponto representa uma variante. O eixo horizontal mostra a mudança na taxa
de sucesso dos óstios e o vertical mostra a mudança no Dice médio. A posição
em relação ao zero evidencia se o parâmetro afetou uma ou ambas as etapas.

In [ ]:
tradeoff_df = relative_effect_df.loc[
    ~relative_effect_df["variant"].eq("baseline")
].copy()

fig, ax = plt.subplots(figsize=(9, 6), constrained_layout=True)
ax.axhline(0, color="#777777", linewidth=1)
ax.axvline(0, color="#777777", linewidth=1)
ax.scatter(
    tradeoff_df["delta_ostia_success_pp"],
    tradeoff_df["delta_dice_vs_baseline"],
    color="#31688e",
    s=65,
)
for row in tradeoff_df.itertuples(index=False):
    ax.annotate(
        row.variant,
        (row.delta_ostia_success_pp, row.delta_dice_vs_baseline),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_xlabel("Variação no sucesso dos óstios (p.p.)", fontsize=12)
ax.set_ylabel("Variação no Dice médio", fontsize=12)
ax.set_title("Sensibilidade da localização e da segmentação", fontsize=13)
plt.show()